# DQN tự viết trên CartPole

CartPole-v1: giữ một cây sào thẳng đứng trên xe đẩy. Observation 4 số thực (vị trí, vận tốc xe, góc, vận tốc góc), 2 action
(đẩy trái, đẩy phải), reward +1 mỗi bước còn đứng, episode kết thúc khi sào nghiêng quá 12° hoặc xe ra khỏi khung
(`terminated`) hoặc sau 500 bước (`truncated`). Return tối đa là 500.

Bạn thay bảng Q của bài Basic 02 bằng một MLP và thêm hai mảnh làm DQN train được: **replay buffer** và **target network**
(mục 4 lý thuyết), rồi đo xem bỏ từng mảnh thì chuyện gì xảy ra.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm, lớp. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) chạy env thật, train, vẽ; test bỏ qua chúng.
- Chấm: `uv run pytest tracks/ai-engineer/modules/07-reinforcement-learning/exercises/intermediate/01-dqn-cartpole -q`

In [ ]:
# explore: giới hạn luồng, tìm gốc repo
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")   # MLP nhỏ: nhiều luồng hơn không nhanh hơn, chỉ làm máy chậm
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
print(ROOT)

In [ ]:
import copy
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

## 1. Replay buffer

`ReplayBuffer(capacity, obs_dim, seed)` lưu chuyển `(obs, action, reward, next_obs, terminated)` vào các mảng numpy cấp
sẵn kích thước `capacity` (obs `float32`, action `int64`, reward và terminated `float32`). Đầy thì ghi đè chuyển **cũ nhất**
(vòng tròn). `sample(batch_size)` lấy chỉ số đều, có hoàn lại, trong phần đã ghi, bằng `self.rng = np.random.default_rng(seed)`,
và trả dict tensor `obs (B, obs_dim)`, `actions (B,)`, `rewards (B,)`, `next_obs (B, obs_dim)`, `terminated (B,)`.

Chỉ lưu `terminated`, không lưu `truncated`: time limit không phải trạng thái kết thúc (mục 1 lý thuyết).

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity: int, obs_dim: int, seed: int):
        # TODO: cấp sẵn mảng numpy cho obs, next_obs (float32), actions (int64), rewards, terminated (float32);
        # con trỏ ghi, số phần tử đã có; self.rng = np.random.default_rng(seed)
        raise NotImplementedError

    def add(self, obs, action: int, reward: float, next_obs, terminated: bool) -> None:
        # TODO: ghi vào vị trí con trỏ, đầy thì vòng về đầu (ghi đè chuyển cũ nhất)
        raise NotImplementedError

    def __len__(self) -> int:
        # TODO: số chuyển đang có (tối đa capacity)
        raise NotImplementedError

    def sample(self, batch_size: int) -> dict[str, torch.Tensor]:
        # TODO: chỉ số đều có hoàn lại trong phần đã ghi, lấy từ self.rng; trả dict tensor
        raise NotImplementedError

## 2. Q-network, ε-greedy

- `QNetwork(obs_dim, n_actions, hidden)`: `Linear → ReLU → Linear → ReLU → Linear`, hai tầng ẩn rộng `hidden`, output là
  Q của mọi action, shape `(B, n_actions)`.
- `linear_epsilon(step, start, end, decay_steps)`: như bài Basic 02.
- `select_action(q_net, obs, epsilon, rng, n_actions)`: với xác suất ε (`rng.random() < epsilon`) trả `int(rng.integers(n_actions))`,
  ngược lại `argmax` Q của `obs` (một observation numpy shape `(obs_dim,)`), không ghi graph autograd. Trả `int`.

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, obs_dim: int, n_actions: int, hidden: int = 128):
        super().__init__()
        # TODO: MLP hai tầng ẩn, ReLU
        raise NotImplementedError

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        # TODO: (B, obs_dim) -> (B, n_actions)
        raise NotImplementedError


def linear_epsilon(step: int, start: float, end: float, decay_steps: int) -> float:
    # TODO
    raise NotImplementedError


def select_action(q_net: nn.Module, obs: np.ndarray, epsilon: float, rng: np.random.Generator, n_actions: int) -> int:
    # TODO
    raise NotImplementedError

## 3. TD target và loss

- `td_targets(batch, target_net, gamma, online_net=None)`: tensor `(B,)`, không mang gradient.
  - DQN (`online_net=None`): $$y = r + \gamma\,(1 - d)\,\max_{a'} Q_{\text{target}}(s', a')$$.
  - Double DQN (có `online_net`): action chọn bằng mạng online, giá trị đọc từ target:
    $$y = r + \gamma\,(1 - d)\,Q_{\text{target}}\big(s', \arg\max_{a'} Q_{\text{online}}(s', a')\big)$$.
- `dqn_loss(batch, q_net, target_net, gamma, double=False)`: Huber loss (`F.smooth_l1_loss`, trung bình) giữa
  $$Q_{\text{online}}(s, a)$$ của đúng action đã làm và target. Gradient chỉ đi vào `q_net`.

In [ ]:
def td_targets(batch: dict, target_net: nn.Module, gamma: float, online_net: nn.Module | None = None) -> torch.Tensor:
    # TODO: shape (B,), không mang gradient; online_net khác None là Double DQN
    raise NotImplementedError


def dqn_loss(batch: dict, q_net: nn.Module, target_net: nn.Module, gamma: float, double: bool = False) -> torch.Tensor:
    # TODO: Huber loss giữa Q(s, a) của action đã làm và td_targets
    raise NotImplementedError

## 4. Đánh giá và vòng train

`evaluate_policy(policy, make_env, n_episodes, seed)`: tạo **một** env bằng `make_env()`, chạy `n_episodes` episode, episode
thứ `i` reset với `seed + i`, action là `policy(obs)`; trả mảng numpy tổng reward từng episode. Cùng seed, cùng policy tất
định thì cùng kết quả.

`train_dqn(make_env, total_steps, seed, **config)` theo đúng các bước sau, trả dict:

1. `torch.manual_seed(seed)`, `rng = np.random.default_rng(seed)`; một env; `q_net` và `target_net` cùng trọng số ban đầu;
   Adam với `lr`; `ReplayBuffer(buffer_size, obs_dim, seed)`; `env.reset(seed=seed)`, episode thứ k sau đó reset với `seed + k`.
2. Mỗi bước `step` (0 tới `total_steps - 1`): ε = `linear_epsilon(step, eps_start, eps_end, eps_decay_steps)`, chọn action,
   `env.step`, lưu chuyển với cờ `terminated`, hết episode (terminated hoặc truncated) thì ghi return và số bước, reset.
3. Khi `step >= learning_starts` và `step % train_freq == 0`: lấy batch, `dqn_loss` (với `use_target=False` thì mạng target
   chính là `q_net`), clip norm gradient 10, một bước Adam; ghi loss. Nếu `use_target` và `tau < 1`: ngay sau bước Adam,
   cập nhật mềm (Polyak) mọi tham số $$\theta' \leftarrow (1 - \tau)\theta' + \tau\theta$$, không ghi graph autograd.
4. Nếu `use_target` và `tau >= 1`: khi `step % target_update == 0`, chép trọng số `q_net` sang `target_net` (hard update).
5. Khi `eval_every > 0` và `(step + 1) % eval_every == 0`: `evaluate_policy` greedy (ε = 0) với `eval_episodes` episode, seed
   `eval_seed`; ghi `(step + 1, mean)`; giữ bản sao `state_dict` tốt nhất (bằng hoặc hơn thì thay).

Trả `{"returns": [...], "end_steps": [...], "losses": [...], "evals": [(step, mean), ...], "q_net": q_net,
"best_state": state_dict tốt nhất hoặc None}`.

In [ ]:
def evaluate_policy(policy, make_env, n_episodes: int, seed: int) -> np.ndarray:
    # TODO
    raise NotImplementedError


def train_dqn(make_env, total_steps: int, seed: int, *, buffer_size: int = 50_000, batch_size: int = 128,
              gamma: float = 0.99, lr: float = 1e-4, learning_starts: int = 1_000, train_freq: int = 1,
              tau: float = 0.005, target_update: int = 500, eps_start: float = 1.0, eps_end: float = 0.05,
              eps_decay_steps: int = 10_000, double: bool = False, use_target: bool = True, hidden: int = 128,
              eval_every: int = 5_000, eval_episodes: int = 10, eval_seed: int = 100_000) -> dict:
    # TODO: theo đúng 5 bước ở trên
    raise NotImplementedError

In [ ]:
# explore: làm quen CartPole, mốc của policy ngẫu nhiên
import gymnasium as gym

make_env = lambda: gym.make("CartPole-v1")
env = make_env()
print(env.observation_space, env.action_space, "| time limit:", env.spec.max_episode_steps)
rng = np.random.default_rng(0)
random_returns = evaluate_policy(lambda o: int(rng.integers(2)), make_env, 100, 10_000)
print(f"policy ngẫu nhiên, 100 episode: return {random_returns.mean():.1f} ± {random_returns.std():.1f}")

In [ ]:
# explore: train 3 cấu hình x 3 seed, 50 000 bước env mỗi lần (khoảng 80-90 giây trên CPU), cả cell khoảng 13 phút
torch.set_num_threads(4)
CONFIGS = {"DQN (soft update)": {}, "hard update mỗi 500 bước": {"tau": 1.0}, "không target net": {"use_target": False}}
runs = {}
for name, cfg in CONFIGS.items():
    for seed in [0, 1, 2]:
        start = time.perf_counter()
        runs[name, seed] = train_dqn(make_env, 50_000, seed, **cfg)
        evals = [int(v) for _, v in runs[name, seed]["evals"]]
        print(f"{name:25s} seed {seed}: {time.perf_counter() - start:4.0f} s | {len(runs[name, seed]['returns'])} episode"
              f" | eval greedy mỗi 5k bước: {evals}")

In [ ]:
# explore: đường cong học (trung bình trượt 20 episode theo số bước env)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(CONFIGS), figsize=(15, 3.5), sharey=True)
for ax, name in zip(axes, CONFIGS):
    for seed in [0, 1, 2]:
        h = runs[name, seed]
        r = np.convolve(h["returns"], np.ones(20) / 20, mode="valid")
        ax.plot(h["end_steps"][19:], r, label=f"seed {seed}")
    ax.set_title(name); ax.set_xlabel("bước env"); ax.axhline(500, color="gray", lw=0.5)
axes[0].set_ylabel("return (TB trượt 20)"); axes[0].legend()
plt.show()

In [ ]:
# explore: đánh giá cuối trên 20 episode cố định (seed 10 000..10 019), trọng số cuối và checkpoint tốt nhất
def greedy_of(net):
    return lambda o: int(net(torch.as_tensor(o, dtype=torch.float32).unsqueeze(0)).argmax())

print(f"{'cấu hình':25s} | {'trọng số cuối (3 seed)':28s} | checkpoint tốt nhất (3 seed)")
for name in CONFIGS:
    final, best = [], []
    for seed in [0, 1, 2]:
        h = runs[name, seed]
        final.append(evaluate_policy(greedy_of(h["q_net"]), make_env, 20, 10_000).mean())
        net = QNetwork(4, 2)
        net.load_state_dict(h["best_state"])
        best.append(evaluate_policy(greedy_of(net), make_env, 20, 10_000).mean())
    print(f"{name:25s} | {' / '.join(f'{v:5.1f}' for v in final):28s} | {' / '.join(f'{v:5.1f}' for v in best)}")

In [ ]:
# explore: Q dự đoán ở trạng thái đầu so với return chiết khấu thật của policy greedy (overestimation, mục 4 lý thuyết)
def q_vs_mc(net, n=20, gamma=0.99):
    env = make_env()
    q_pred, mc = [], []
    for i in range(n):
        obs, _ = env.reset(seed=10_000 + i)
        with torch.no_grad():
            q_pred.append(net(torch.as_tensor(obs).unsqueeze(0)).max().item())
        rewards, done = [], False
        while not done:
            obs, r, te, tr, _ = env.step(greedy_of(net)(obs))
            rewards.append(r)
            done = te or tr
        mc.append(sum(r * gamma**k for k, r in enumerate(rewards)))
    return np.mean(q_pred), np.mean(mc)

for name in CONFIGS:
    pairs = [q_vs_mc(runs[name, s]["q_net"]) for s in [0, 1, 2]]
    print(f"{name:25s} | max Q(s0) dự đoán: {' / '.join(f'{q:8.1f}' for q, _ in pairs)}"
          f" | return chiết khấu thật: {' / '.join(f'{m:5.1f}' for _, m in pairs)}")

## 5. Đọc kết quả

So với bảng mục 4 lý thuyết. Ba điều cần thấy: bỏ target network thì Q-value nổ và policy sụp; đường cong DQN dao động mạnh
ngay cả khi đã chạm 500 (vì vậy cần đánh giá định kỳ và giữ checkpoint tốt nhất thay vì lấy trọng số cuối); Q dự đoán so với
return thật cho biết mức overestimation. Số của bạn có thể lệch nhẹ nếu thứ tự lấy số ngẫu nhiên khác bản tham chiếu, nhưng
kết luận phải giống.

Sau khi chạy hết notebook, làm phần **trắc nghiệm** của bài trong app.